In [2]:
import os
import sys
import math
import locale
import pypdf
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from typing import List
from pydantic import BaseModel, Field
from utils_ccd import get_connection, get_info_file_path, extract_text_from_pdf, generate_pdf_office
from datetime import datetime
from dotenv import load_dotenv
from docxtpl import DocxTemplate

load_dotenv()
conn = get_connection()
locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')

from pathlib import Path
from langchain_openai import AzureOpenAI, AzureChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate
from docx2pdf import convert

llm = AzureChatOpenAI(model_name="gpt-4o")

c:\Users\05911205424\Documents\Dev\ccd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_planilha = pd.read_excel('db/descontos_folha.xlsx')

In [4]:
df_planilha

,processo,cpf,n_parcelas,obs,orgao,cadastrado,finalizado,valor_total,transf_frap
0,004404/2024,72114673472,2.0,NaN,Serviço Autônomo de Água e esgoto de Ceará Mirim,SIM,sim,458.21,sim
1,003020/2022,834517493,NaN,NaN,NaN,SIM,NaN,84443.57,NaN
2,000423/2024,2140437462,1.0,Perdeu vínculo,NaN,NaN,NaN,21474.79,NaN
3,003287/2023,22136967487,16.0,NaN,IPERN,SIM,NaN,19455.48,NaN
4,002027/2024,1075318440,30.0,NaN,SESAP/RN,SIM,NaN,28601.96,NaN
5,003366/2024,27170390400,4.0,NaN,IDEMA,SIM,sim,5634.33,NaN
6,003392/2024,81357427468,15.0,NaN,ALRN,SIM,NaN,8420.67,NaN
7,000415/2024,8984449423,33.0,NaN,IPSS - Inst. Prev. SOCIAL Dos Serv Mun. de Olh...,SIM,NaN,6717.34,NaN
8,002611/2024,13070231420,40.0,Nulidade da certidão de trânsito em julgado. C...,NaN,NaN,NaN,62601.71,NaN
9,003347/2024,8548536420,12.0,NaN,IPERN,SIM,NaN,2965.32,NaN


In [6]:
processos = ', '.join(df_planilha['processo'].apply(lambda x: f"'{x}'").tolist())
sql_processos = ''' 
SELECT CONCAT(p.numero_processo, '/', p.ano_processo) as processo,
setor_atual
FROM processo.dbo.Processos p
WHERE CONCAT(p.numero_processo, '/', p.ano_processo) IN ({processos})
'''
df_processos = pd.read_sql(sql_processos.format(processos=processos), conn)

In [8]:
df_processos.to_excel('db/localizacao_processos_folha.xlsx', index=False)